Using the Boston data set, fit classification models in order to predict
whether a given suburb has a crime rate above or below the median.
Explore logistic regression, LDA, naive Bayes, and KNN models using
various subsets of the predictors. Describe your findings.
Hint: You will have to create the response variable yourself, using the
variables that are contained in the Boston data set

In [3]:
import pandas as pd
import plotly.express as px
from ISLP import load_data
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV

boston = load_data('Boston')
boston.columns = boston.columns.str.lower()
boston = boston.astype(float)
boston.head()

# Response var
boston['high_crime'] = (boston['crim'] > boston['crim'].median()).astype(int)
predictors = ['zn', 'indus', 'chas', 'nox', 'rm', 'age', 'dis', 'rad', 'tax', 'ptratio', 'lstat']
X = boston[predictors]
y = boston['high_crime']


*   **CRIM**: Per capita crime rate by town.
*   **ZN**: Proportion of residential land zoned for lots over 25,000 sq.ft.
*   **INDUS**: Proportion of non-retail business acres per town.
*   **CHAS**: Charles River dummy variable (1 if tract bounds river; 0 otherwise).
*   **NOX**: Nitric oxides concentration (parts per 10 million).
*   **RM**: Average number of rooms per dwelling.
*   **AGE**: Proportion of owner-occupied units built prior to 1940.
*   **DIS**: Weighted distances to five Boston employment centers.
*   **RAD**: Index of accessibility to radial highways.
*   **TAX**: Full-value property tax rate per $10,000.
*   **PTRATIO**: Pupil-teacher ratio by town.
*   **B**: $1000(Bk - 0.63)^2$ where Bk is the proportion of Black residents by town.
*   **LSTAT**: % Lower status of the population.
*   **MEDV**: Median value of owner-occupied homes in $1000s 

# LDA

In [4]:
pipe_lda = Pipeline([
    ('scaler', StandardScaler()),
    ('lda', LinearDiscriminantAnalysis())
])
pipe_lda.fit(X_train, y_train)
y_pred_lda = pipe_lda.predict(X_test)

print('LDA accuracy:', accuracy_score(y_test, y_pred_lda))
print(classification_report(y_test, y_pred_lda))

NameError: name 'X_train' is not defined

In [ ]:
import numpy as np
import plotly.express as px

crim = boston['crim'].dropna()

fig = px.histogram(crim, nbins=30, title='CRIM', labels={'value': 'CRIM'})
fig.update_layout(xaxis_title='CRIM', yaxis_title='Count')
fig.show()

fig2 = px.histogram(np.log1p(crim), nbins=30, title='log1p(CRIM)', labels={'value': 'log1p(CRIM)'})
fig2.update_layout(xaxis_title='log1p(CRIM)', yaxis_title='Count')
fig2.show()

# Naive Bayes


In [ ]:
pipe_nb = Pipeline([
    ('scaler', StandardScaler()),
    ('nb', GaussianNB())
])
pipe_nb.fit(X_train, y_train)
y_pred_nb = pipe_nb.predict(X_test)

print('Naive Bayes accuracy:', accuracy_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb))

Naive Bayes accuracy: 0.8552631578947368
              precision    recall  f1-score   support

           0       0.84      0.88      0.86        76
           1       0.88      0.83      0.85        76

    accuracy                           0.86       152
   macro avg       0.86      0.86      0.86       152
weighted avg       0.86      0.86      0.86       152



In [ ]:
import plotly.express as px

corr = X[predictors].corr()

fig = px.imshow(corr,
                text_auto=True,
                color_continuous_scale='RdBu',
                title='Predictor Correlation Matrix')
fig.update_layout(width=700, height=600)
fig.show()

# Logistic regression

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=1, stratify=y
)

pipe_log = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(max_iter=2000, random_state=1))
])
pipe_log.fit(X_train, y_train)
y_pred_log = pipe_log.predict(X_test)

print('Logistic regression accuracy:', accuracy_score(y_test, y_pred_log))
print(classification_report(y_test, y_pred_log))

Logistic regression accuracy: 0.9078947368421053
              precision    recall  f1-score   support

           0       0.87      0.96      0.91        76
           1       0.96      0.86      0.90        76

    accuracy                           0.91       152
   macro avg       0.91      0.91      0.91       152
weighted avg       0.91      0.91      0.91       152



# KNN

In [ ]:
pipe_knn = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])
param_grid = {'knn__n_neighbors': list(range(1, 16))}
search_knn = GridSearchCV(pipe_knn, param_grid, cv=5)
search_knn.fit(X_train, y_train)

best_knn = search_knn.best_estimator_
y_pred_knn = best_knn.predict(X_test)

print('Best KNN k:', search_knn.best_params_['knn__n_neighbors'])
print('KNN accuracy:', accuracy_score(y_test, y_pred_knn))
print(classification_report(y_test, y_pred_knn))

Best KNN k: 4
KNN accuracy: 0.9013157894736842
              precision    recall  f1-score   support

           0       0.86      0.96      0.91        76
           1       0.96      0.84      0.90        76

    accuracy                           0.90       152
   macro avg       0.91      0.90      0.90       152
weighted avg       0.91      0.90      0.90       152



In [ ]:
import plotly.graph_objects as go

results = search_knn.cv_results_
k_values = list(range(1, 16))
mean_scores = results['mean_test_score']
std_scores = results['std_test_score']
best_k = search_knn.best_params_['knn__n_neighbors']

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=k_values,
    y=mean_scores,
    mode='lines+markers',
    name='CV Accuracy',
    line=dict(color='#1f77b4', width=2),
    marker=dict(size=7),
    error_y=dict(
        type='data',
        array=std_scores,
        visible=True,
        color='lightblue',
        thickness=1.5
    )
))

fig.add_vline(
    x=best_k,
    line_dash='dash',
    line_color='red',
    annotation_text=f'Best K={best_k}',
    annotation_position='top right'
)

fig.update_layout(
    title='KNN Cross-Validation Accuracy vs K',
    xaxis=dict(title='K (number of neighbors)', tickmode='linear', dtick=1),
    yaxis=dict(title='Mean CV Accuracy', tickformat='.3f'),
    hovermode='x unified'
)

fig.show()

In [ ]:
models = ['Logistic Regression', 'LDA', 'Naive Bayes', 'KNN']
accuracies = [
    accuracy_score(y_test, y_pred_log),
    accuracy_score(y_test, y_pred_lda),
    accuracy_score(y_test, y_pred_nb),
    accuracy_score(y_test, y_pred_knn)
]

fig = px.bar(
    x=models,
    y=accuracies,
    labels={'x': 'Model', 'y': 'Accuracy'},
    title='Boston Crime Classification: Model Performance Comparison',
    color=models,
    color_discrete_sequence=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
)
fig.update_layout(showlegend=False, yaxis=dict(range=[0.8, 1.0]))
fig.update_traces(textposition='outside')
fig.show()